# 📰 News Embedding Worker
Embeds news articles using **Qwen3-Embedding-8B** on Kaggle T4 GPU and saves vectors to PostgreSQL via n8n webhooks.

---
### ⚠️ Before running this notebook:
1. Attach the `qwen3-embedding-8b-model` dataset (right panel → Input → Add Input)
2. Set Accelerator to **GPU T4 x2** (right panel)
3. Turn Internet **ON** (right panel)
4. Fill in your credentials in **Cell 2** below
5. Run all cells top to bottom: **Cell 1 → 2 → 3 → 4**
---

In [ ]:
# ============================================================
# CELL 1 — Install dependencies
# Run once per session. Takes ~1 minute.
# ============================================================
import subprocess, sys, os

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "sentence-transformers"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U",
                       "bitsandbytes>=0.46.1"])

import bitsandbytes as bnb
print(f"✅ bitsandbytes version: {bnb.__version__}")
print("✅ Dependencies installed")

---
## ✏️ Cell 2 — Fill in your credentials here

In [ ]:
# ============================================================
# CELL 2 — Configuration
# Edit the values below before running.
# ============================================================
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# --- n8n webhook URLs (get these from your n8n instance) ---
N8N_GET_URL  = "https://n8n.3rfan.ir/webhook/4f1d52bf-25d5-4e0b-ab30-123f680d0265"     # e.g. https://n8n.yourdomain.com/webhook/xxx
N8N_SAVE_URL = "https://n8n.3rfan.ir/webhook/4f1d52bf-25d5-4e0b-ab30-123f680d0255"  # e.g. https://n8n.yourdomain.com/webhook/yyy
N8N_API_KEY  = "mer30kehasti"               # the X-API-Key secret set in n8n

# --- Worker identity (use a unique name if running multiple workers) ---
NODE_NAME    = "kaggle-t4-worker"

# --- Performance settings ---
BATCH_SIZE   = 1      # articles fetched per cycle (keep at 1 to avoid OOM)
MAX_HOURS    = 8.5    # stop before Kaggle's 9h session limit

# --- Model path (do not change if dataset is attached correctly) ---
import glob
matches = glob.glob("/kaggle/input/**/qwen3-embedding", recursive=True)
if matches:
    MODEL_PATH = matches[0]
    print(f"✅ Model found at: {MODEL_PATH}")
else:
    MODEL_PATH = "/kaggle/input/datasets/YOUR_USERNAME/qwen3-embedding-8b-model/qwen3-embedding"
    print(f"⚠️  Model not auto-detected. Set MODEL_PATH manually: {MODEL_PATH}")

print(f"\nConfig loaded:")
print(f"  NODE_NAME  : {NODE_NAME}")
print(f"  BATCH_SIZE : {BATCH_SIZE}")
print(f"  MAX_HOURS  : {MAX_HOURS}h")
print(f"  N8N_GET    : {N8N_GET_URL[:40]}...")

---
## 🤖 Cell 3 — Load Model
Takes 2–3 minutes. Uses 8-bit quantization (~8.2 GB VRAM).

In [ ]:
# ============================================================
# CELL 3 — Load Qwen3-Embedding-8B model (8-bit quantized)
# ============================================================
from sentence_transformers import SentenceTransformer
from transformers import BitsAndBytesConfig
import torch

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"GPU memory before load: {torch.cuda.memory_allocated(0)/1e9:.1f} GB")

quant_cfg = BitsAndBytesConfig(load_in_8bit=True)

model = SentenceTransformer(
    MODEL_PATH,
    device=device,
    model_kwargs={"quantization_config": quant_cfg}
)

used = torch.cuda.memory_allocated(0) / 1e9
free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1e9
print(f"GPU used: {used:.1f} GB  |  free: {free:.1f} GB")

# Verify model output dimensions
test_vector = model.encode(["stock market earnings report"], normalize_embeddings=True)
print(f"✅ Model ready — embedding shape: {test_vector.shape}")

if test_vector.shape[1] != 4096:
    print(f"⚠️  WARNING: Expected 4096 dimensions, got {test_vector.shape[1]}")
else:
    print("✅ Dimensions correct: 4096")

---
## 🚀 Cell 4 — Start Embedding
Runs until all articles are embedded or `MAX_HOURS` is reached.

Progress is saved automatically — if the session stops, just run again and it continues from where it left off.

In [ ]:
# ============================================================
# CELL 4 — Main embedding loop
# Fetches articles from n8n → embeds → saves vectors back.
# Safe to stop and resume at any time.
# ============================================================
import requests
import time
import torch

headers  = {"X-API-Key": N8N_API_KEY, "Content-Type": "application/json"}
total    = 0
errors   = 0
start    = time.time()

print(f"Worker  : {NODE_NAME}")
print(f"Batch   : {BATCH_SIZE}")
print(f"Max time: {MAX_HOURS}h")
print("-" * 60)

while True:

    # --- Time limit ---
    elapsed_h = (time.time() - start) / 3600
    if elapsed_h >= MAX_HOURS:
        print(f"\n⏱️  Time limit reached ({MAX_HOURS}h). Stopping cleanly.")
        print(f"Total embedded this session: {total:,}")
        break

    # --- Get batch from n8n ---
    try:
        resp = requests.post(
            N8N_GET_URL,
            json={"batch_size": BATCH_SIZE, "node_name": NODE_NAME},
            headers=headers,
            timeout=30
        )
        records = resp.json().get("records", [])
    except Exception as e:
        print(f"⚠️  GET error: {e}")
        errors += 1
        time.sleep(10)
        continue

    if not records:
        print(f"\n✅ No more pending articles. All done!")
        print(f"Total embedded this session: {total:,}")
        break

    ids   = [r["id"] for r in records]
    texts = [r.get("description") or r.get("title") or "" for r in records]

    # --- Embed ---
    try:
        t0      = time.time()
        vectors = model.encode(
            texts,
            batch_size=1,
            normalize_embeddings=True,
            show_progress_bar=False
        )
        speed = len(texts) / (time.time() - t0)
        torch.cuda.empty_cache()

    except RuntimeError as e:
        torch.cuda.empty_cache()
        print(f"⚠️  Embed error (id={ids}): {str(e)[:80]}")
        errors += 1
        time.sleep(5)
        continue

    # --- Save vectors to n8n ---
    try:
        payload = {"vectors": [
            {"id": ids[i], "vector": vectors[i].tolist(), "node_name": NODE_NAME}
            for i in range(len(ids))
        ]}
        requests.post(N8N_SAVE_URL, json=payload, headers=headers, timeout=60)
    except Exception as e:
        print(f"⚠️  SAVE error: {e}")
        errors += 1
        time.sleep(5)
        continue

    total     += len(records)
    elapsed_m  = (time.time() - start) / 60

    print(
        f"✅ {total:>8,} embedded | "
        f"{speed:5.1f} art/sec | "
        f"{elapsed_m:6.1f} min | "
        f"{elapsed_h:.2f}h / {MAX_HOURS}h | "
        f"errors: {errors}"
    )